## **Download the dataset from official site**

In [1]:
#--------- Download day 1 data ----------#
!wget http://cicresearch.ca/CICDataset/CICDDoS2019/Dataset/CSVs/CSV-01-12.zip

--2025-06-11 08:14:28--  http://cicresearch.ca/CICDataset/CICDDoS2019/Dataset/CSVs/CSV-01-12.zip
Resolving cicresearch.ca (cicresearch.ca)... 205.174.165.80
Connecting to cicresearch.ca (cicresearch.ca)|205.174.165.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2330434641 (2.2G) [application/zip]
Saving to: ‘CSV-01-12.zip’

CSV-01-12.zip       100%[===================>]   2.17G  7.43MB/s    in 3m 21s  

2025-06-11 08:17:50 (11.0 MB/s) - ‘CSV-01-12.zip’ saved [2330434641/2330434641]



In [2]:
#--------- Download day 2 data ----------#
!wget http://cicresearch.ca/CICDataset/CICDDoS2019/Dataset/CSVs/CSV-03-11.zip

--2025-06-11 08:17:50--  http://cicresearch.ca/CICDataset/CICDDoS2019/Dataset/CSVs/CSV-03-11.zip
Resolving cicresearch.ca (cicresearch.ca)... 205.174.165.80
Connecting to cicresearch.ca (cicresearch.ca)|205.174.165.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 918815761 (876M) [application/zip]
Saving to: ‘CSV-03-11.zip’

CSV-03-11.zip       100%[===================>] 876.25M  21.3MB/s    in 63s     

2025-06-11 08:18:53 (14.0 MB/s) - ‘CSV-03-11.zip’ saved [918815761/918815761]



In [3]:
#------------- Import Required Libraries ----------#
import zipfile
import os
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn import preprocessing
import warnings
from pandas.errors import DtypeWarning
warnings.filterwarnings("ignore", category=DtypeWarning)

In [4]:
#--------- Unzip the zipfiles -----------#

# Unzip day 1 data
with zipfile.ZipFile('CSV-01-12.zip', 'r') as zip_ref:
    zip_ref.extractall('.')
os.remove('CSV-01-12.zip')

# Unzip day 2 data
with zipfile.ZipFile('CSV-03-11.zip', 'r') as zip_ref:
    zip_ref.extractall('.')
os.remove('CSV-03-11.zip')

## **Read the CSV files**

In [ ]:
#---------- Helper functions to read csv files ------#

mult = 5

def load_file(path):
  data = pd.read_csv(path, sep = ',')
  is_benign = data[' Label'] == 'BENIGN'
  flows_ok = data[is_benign]
  flows_ddos_full = data[~is_benign]

  sizeDownSample = len(flows_ok)*mult

  if (len(flows_ok)*mult) < (len(flows_ddos_full)):
    flows_ddos_reduced = resample(flows_ddos_full,
                                  replace = False,
                                  n_samples = sizeDownSample,
                                  random_state = 42)
  else:
    flows_ddos_reduced = flows_ddos_full

  return flows_ok, flows_ddos_reduced

def load_huge_file(path):
  df_chunk = pd.read_csv(path, chunksize = 500000)
  chunk_list_ok = []
  chunk_list_ddos = []

  for chunk in df_chunk:
    is_benign = chunk[' Label'] == 'BENIGN'
    flows_ok = chunk[is_benign]
    flows_ddos_full = chunk[~is_benign]

    if (len(flows_ok)*mult) < (len(flows_ddos_full)):
      sizeDownSample = len(flows_ok)*mult

      flows_ddos_reduced = resample(flows_ddos_full,
                                    replace = False,
                                    n_samples = sizeDownSample,
                                    random_state = 42)
    else:
      flows_ddos_reduced = flows_ddos_full

    chunk_list_ok.append(flows_ok)
    chunk_list_ddos.append(flows_ddos_reduced)

  flows_ok = pd.concat(chunk_list_ok)
  flows_ddos = pd.concat(chunk_list_ddos)

  return flows_ok, flows_ddos

In [ ]:
#----------- Read all files one by one -----------#

# File 1 - TFTP
flows_ok, flows_ddos = load_huge_file('01-12/TFTP.csv')
print("1. TFTP.csv loaded..")

# File 2 - DrDoS_LDAP
a, b = load_file('01-12/DrDoS_LDAP.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("2. DrDoS_LDAP.csv loaded..")

# File 3 - DrDoS_MSSQL
a, b = load_file('01-12/DrDoS_MSSQL.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("3. DrDoS_MSSQL.csv loaded..")

# File 4 - DrDoS_NetBIOS
a, b = load_file('01-12/DrDoS_NetBIOS.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("4. DrDoS_NetBIOS.csv loaded..")

# File 5 - DrDoS_NTP
a, b = load_file('01-12/DrDoS_NTP.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("5. DrDoS_NTP.csv loaded..")

# File 6 - DrDoS_SNMP
a, b = load_file('01-12/DrDoS_SNMP.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("6. DrDoS_SNMP.csv loaded..")

# File 7 - DrDoS_SSDP
a, b = load_file('01-12/DrDoS_SSDP.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("7. DrDoS_SSDP.csv loaded..")

# File 8 - DrDoS_UDP
a, b = load_file('01-12/DrDoS_UDP.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("8. DrDoS_UDP.csv loaded..")

# File 9 - Syn
a, b = load_file('01-12/Syn.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("9. Syn.csv loaded..")

# File 10 - DrDoS_DNS
a, b = load_file('01-12/DrDos_DNS.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("10. DrDos_DNS.csv loaded..")

# File 11 - UDPLag
a, b = load_file('01-12/UDPLag.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("11. UDPLag.csv loaded..")

# File 12 - LDAP
a, b = load_file('cicddos2019/03-11/LDAP.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("12. LDAP.csv loaded..")

# File 13 - MSSQL
a, b = load_file('cicddos2019/03-11/MSSQL.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("13. MSSQL.csv loaded..")

# File 14 - NetBIOS
a, b = load_file('cicddos2019/03-11/NetBIOS.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("14. NetBIOS.csv loaded..")

# File 15 - PortMap
a, b = load_file('cicddos2019/03-11/PortMap.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("15. PortMap.csv loaded..")

# File 16 - Syn
a, b = load_file('cicddos2019/03-11/Syn.csv')
flows_ok = pd.concat([flows_ok, a], ignore_index=True)
flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
print("16. Syn.csv loaded..")

# Cleanup and Save
del a, b
samples = pd.concat([flows_ok, flows_ddos], ignore_index=True)
samples.to_csv('all-data.csv', index=False)
print("✅ All files merged and saved to 'all-data.csv'")
